# 파인튜닝 실행 분석

`reports/current/v4/finetune_runs.jsonl`에 쌓인 파인튜닝 실행들을 읽어 네 가지를 확인합니다.

1. **seed 편차** — 모델 크기 효과가 seed 흔들림과 구분되는가
2. **오답 겹침** — 실행들이 같은 문항에서 틀리는가, 다른 문항에서 틀리는가
3. **전원 오답** — 모든 모델이 틀리는 문항은 무엇인가
4. **임계값 스윕** — 표를 확신도로 써서 놓침과 과잉의 균형점을 고른다

학습은 하지 않습니다. 이미 저장된 예측만 다시 읽으므로 GPU가 필요 없습니다.

`sklearn` 없이 돌도록 macro F1은 직접 구현했습니다. `pandas`와 `matplotlib`만 있으면 됩니다.


In [ ]:
from pathlib import Path
import json
from collections import Counter
from itertools import combinations

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

RUNS = ROOT / 'reports' / 'current' / 'v4' / 'finetune_runs.jsonl'
DATASET = ROOT / 'data' / 'labels' / 'label_dataset_v4.jsonl'
ACCEPT = '통상수용'

records = [json.loads(line) for line in RUNS.open(encoding='utf-8')]
print(f'{len(records)}개 실행 기록')


## 실행 목록

`fold -1`(10 fold 전체)로 돌린 것만 씁니다. fold 하나짜리 연기 테스트는 제외합니다.

**fold 평균**은 10개 fold macro F1의 단순 평균이고, **통합 OOF**는 924건 예측을 통째로
모아 한 번에 계산한 값입니다. fold마다 크기와 라벨 분포가 달라 두 값이 다릅니다.


In [ ]:
def macro_f1(gold, pred):
    """라벨 집합 전체에 대한 macro F1. sklearn 없이 계산한다."""
    names = sorted(set(gold) | set(pred))
    total = 0.0
    for name in names:
        tp = sum(1 for g, p in zip(gold, pred) if g == name and p == name)
        fp = sum(1 for g, p in zip(gold, pred) if g != name and p == name)
        fn = sum(1 for g, p in zip(gold, pred) if g == name and p != name)
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        if precision + recall:
            total += 2 * precision * recall / (precision + recall)
    return total / len(names)


def short_name(config):
    family = config['model'].split('/')[-1]
    family = (family.replace('roberta-', 'R')
                    .replace('koelectra-base-v3-discriminator', 'ELEC')
                    .replace('kobigbird-bert-base', 'BIG'))
    parts = [family]
    if config.get('binary'):
        parts.append('2분류')
    if config.get('mask'):
        parts.append('마스킹')
    if config.get('max_length') != 512:
        parts.append(str(config['max_length']))
    parts.append('s' + str(config.get('seed')))
    return ' '.join(parts)


runs = []
for record in records:
    config, results = record['config'], record['results']
    if len(results) < 2:
        continue  # fold 하나짜리 연기 테스트는 뺀다
    predictions = {p['requirement_uid']: (p['gold'], p['pred'])
                   for fold in results for p in fold['predictions']}
    runs.append({
        'name': short_name(config),
        'model': config['model'],
        'seed': config.get('seed'),
        'binary': bool(config.get('binary')),
        'mask': bool(config.get('mask')),
        'max_length': config.get('max_length'),
        'fold_avg': sum(f['test_macro_f1'] for f in results) / len(results),
        'oof': macro_f1([g for g, _ in predictions.values()],
                        [p for _, p in predictions.values()]),
        'predictions': predictions,
    })

summary = pd.DataFrame(runs)[['name', 'seed', 'binary', 'mask', 'max_length',
                              'fold_avg', 'oof']]
summary.round(3)


## 1. seed 편차와 크기 효과

모델을 키우면 좋아지는지 봅니다. **같은 설정을 seed만 바꿔 여러 번 돌린 것이 있어야**
판단할 수 있습니다. 모델 간 차이가 seed 편차보다 작으면 크기 효과라고 말할 수 없습니다.


In [ ]:
plain = [r for r in runs if not r['binary'] and not r['mask'] and r['max_length'] == 512]

by_model = {}
for r in plain:
    by_model.setdefault(r['model'].split('/')[-1], []).append(r['oof'])

variance = pd.DataFrame([
    {'모델': model,
     'n': len(scores),
     '평균': sum(scores) / len(scores),
     '최소': min(scores),
     '최대': max(scores),
     '폭': max(scores) - min(scores)}
    for model, scores in by_model.items()
]).sort_values('평균', ascending=False)

variance.round(3)


seed가 하나뿐인 모델은 `폭`이 0으로 나옵니다. **그 값은 편차를 모른다는 뜻이지**
**안정적이라는 뜻이 아닙니다.** seed를 3개 돌린 모델의 폭과 나란히 놓고 읽어야 합니다.


In [ ]:
multi = variance[variance['n'] >= 3]
if len(multi) >= 2:
    gap = multi['평균'].max() - multi['평균'].min()
    widest = multi['폭'].max()
    print(f'seed 3개 이상인 모델 간 평균 차이  {gap:.3f}')
    print(f'그 모델들의 최대 seed 폭          {widest:.3f}')
    verdict = '구분되지 않는다' if gap < widest else '구분된다'
    print(f'→ 크기 효과는 seed 편차와 {verdict}')
else:
    print('seed 3개 이상인 모델이 둘 미만이라 비교할 수 없다')


## 2. 오답이 겹치는가

앙상블이 이득을 내려면 실행들이 **다른 문항에서** 틀려야 합니다. 똑같이 틀리면 몇 개를
모아도 같은 답이 나옵니다.

Jaccard = 두 실행의 오답 교집합 ÷ 합집합. 1이면 완전히 같고 0이면 전혀 안 겹칩니다.


In [ ]:
three_way = [r for r in runs if not r['binary']]
uids = sorted(set.intersection(*[set(r['predictions']) for r in three_way]))
errors = {r['name']: {u for u in uids if r['predictions'][u][0] != r['predictions'][u][1]}
          for r in three_way}

overlap = pd.DataFrame([
    {'A': a, 'B': b, 'Jaccard': len(ea & eb) / len(ea | eb)}
    for (a, ea), (b, eb) in combinations(errors.items(), 2)
]).sort_values('Jaccard', ascending=False)

print(f'평균 겹침 {overlap["Jaccard"].mean():.2f}')
print()
print('가장 많이 겹치는 쌍')
display(overlap.head(5).round(2))
print('가장 적게 겹치는 쌍')
display(overlap.tail(5).round(2))


같은 seed의 **마스킹·비마스킹 쌍**이 최상위에 오면, 마스킹이 점수뿐 아니라 틀리는 문항까지
바꾸지 못했다는 뜻입니다. 평균값이 우연히 비슷한 것과는 증거의 무게가 다릅니다.

계열이 다른 쌍(`ELEC`, `BIG`)이 하위에 오면 다양성 자체는 확보된 것입니다. 다만 다양성만으로는
부족하고 **비슷하게 강해야** 앙상블에 도움이 됩니다. 약한 모델의 표는 신호가 아니라 잡음입니다.


In [ ]:
# 마스킹 실행은 비마스킹과 거의 같은 오답을 내므로(위 표 참조) 중복으로 세지 않는다.
# 아래 4절의 투표 집합과 같은 9개를 쓴다.
distinct = {r['name']: errors[r['name']]
            for r in three_way if not r['mask']}
hard = set.intersection(*distinct.values())
union = set.union(*distinct.values())
print(f'집계 대상 {len(distinct)}개 실행 (마스킹 제외)')

print(f'전원 오답       {len(hard):4d}건 ({len(hard) / len(uids):.1%})')
print(f'한 번이라도 오답 {len(union):4d}건 ({len(union) / len(uids):.1%})')
print(f'전원 정답       {len(uids) - len(union):4d}건 ({(len(uids) - len(union)) / len(uids):.1%})')
print()
oracle = (len(uids) - len(hard)) / len(uids)
print(f'오라클 상한 (문항마다 맞히는 모델을 고를 수 있다면) 정확도 {oracle:.3f}')


오라클과 실제 다수결의 간극이 크다면, 정보는 모델들 안에 있는데 꺼내지 못하는 것입니다.
다양성 부족이 아니라 **모델이 자기가 언제 맞는지 모르는** 문제이며 캘리브레이션 영역입니다.

`finetune_runs.jsonl`에는 예측 라벨만 있고 확률값이 없어 soft voting은 이 데이터로 잴 수 없습니다.


## 3. 전원이 틀리는 문항

모든 실행이 틀린 문항은 **모델을 바꿔서 해결되지 않는** 부분입니다. 어떤 라벨에 몰려 있는지,
모델이 대신 무엇이라 답하는지 봅니다.


In [ ]:
gold = {u: three_way[0]['predictions'][u][0] for u in uids}
gold_all = Counter(gold.values())
gold_hard = Counter(gold[u] for u in hard)

display(pd.DataFrame([
    {'정답 라벨': label,
     '전원 오답': gold_hard.get(label, 0),
     '전체': count,
     '하드 비율': gold_hard.get(label, 0) / count}
    for label, count in gold_all.most_common()
]).round(3))

guesses = Counter(r['predictions'][u][1] for u in hard for r in three_way)
total = sum(guesses.values())
print('전원 오답 문항에서 모델들이 실제로 답한 것')
for label, n in guesses.most_common():
    print(f'  {label:<12} {n / total:5.1%}')


예측이 다수 클래스(`통상수용`)로 쏠린다면 **모델이 어려우면 안전한 답으로 도망친다는** 뜻입니다.
손실 함수에 `class_weight='balanced'`가 이미 걸려 있으므로(`scripts/modeling/finetune.py`)
가중치로 고칠 수 있는 불균형 문제가 아닙니다.

업무적으로는 이 방향이 가장 나쁩니다. **검토가 필요한데 그냥 받으라고 하는 오류**이기 때문입니다.
반대 방향인 과잉 표시는 사람이 읽고 걸러내면 됩니다.


In [ ]:
rows_by_uid = {r['requirement_uid']: r
               for r in (json.loads(l) for l in DATASET.open(encoding='utf-8'))}

hard_len = pd.Series([len(rows_by_uid[u]['model_text']) for u in hard if u in rows_by_uid])
all_len = pd.Series([len(rows_by_uid[u]['model_text']) for u in uids if u in rows_by_uid])
print(f'원문 길이 중앙값   전원 오답 {hard_len.median():.0f}자 / 전체 {all_len.median():.0f}자')

doc_all = Counter(u.split(':')[0] for u in uids)
doc_hard = Counter(u.split(':')[0] for u in hard)
pd.DataFrame([
    {'문서': d, '전원 오답': doc_hard.get(d, 0), '전체': n, '비율': doc_hard.get(d, 0) / n}
    for d, n in doc_all.most_common()
]).sort_values('비율', ascending=False).round(3)


길이 중앙값이 비슷하면 **잘림이 원인이 아닙니다.** 문서별 비율이 고르게 퍼져 있으면 특정 문서의
수집·라벨링 문제가 아니라 과제 전반의 성질이라는 뜻입니다.


## 4. 임계값 스윕

3분류 예측을 **2분류로 접습니다** — `통상수용` 대 `조치 필요`.

문항마다 "몇 개의 모델이 조치가 필요하다고 했는가"를 셉니다. 이 표 수가 확신도 역할을 합니다.
기준을 낮추면 더 많이 표시하고 덜 놓칩니다.

**여기에는 정답이 없습니다.** 놓침과 과잉의 비용이 다르므로 어느 점을 쓸지는 계산이 아니라
업무 판단입니다.


In [ ]:
voters = [r for r in runs if not r['binary'] and not r['mask']]
n_voters = len(voters)
print(f'투표에 쓰는 실행 {n_voters}개')
print(', '.join(r['name'] for r in voters))

votes = {u: sum(1 for r in voters if r['predictions'][u][1] != ACCEPT) for u in uids}
need = {u for u in uids if gold[u] != ACCEPT}
binary_gold = [0 if gold[u] == ACCEPT else 1 for u in uids]

sweep = []
for k in range(1, n_voters + 1):
    flagged = {u for u in uids if votes[u] >= k}
    tp, fn = len(flagged & need), len(need - flagged)
    sweep.append({
        '기준': f'{k}표 이상',
        '표시': len(flagged),
        '놓침': fn,
        'recall': tp / len(need),
        'precision': tp / len(flagged) if flagged else 0.0,
        '2분류 macro F1': macro_f1(binary_gold,
                                 [1 if votes[u] >= k else 0 for u in uids]),
    })

sweep_df = pd.DataFrame(sweep)
sweep_df.round(3)


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
x = list(range(1, n_voters + 1))
majority = n_voters // 2 + 1

ax1.plot(x, sweep_df['recall'], marker='o', label='recall')
ax1.plot(x, sweep_df['precision'], marker='s', label='precision')
ax1.axvline(majority, color='gray', ls='--', lw=1)
ax1.set_xlabel('threshold (votes)')
ax1.set_ylabel('score')
ax1.set_title('recall vs precision')
ax1.legend()
ax1.grid(alpha=.3)

ax2.plot(x, sweep_df['2분류 macro F1'], marker='o', color='tab:green')
ax2.axvline(majority, color='gray', ls='--', lw=1)
ax2.set_xlabel('threshold (votes)')
ax2.set_ylabel('binary macro F1')
ax2.set_title('binary macro F1')
ax2.grid(alpha=.3)

plt.tight_layout()
plt.show()
print('세로 점선 = 다수결')


### 읽는 법

- **다수결 지점의 값**이 기준선(TF-IDF 2분류 0.788)을 넘는지 먼저 봅니다. 다수결은 아무것도
  고르지 않은 기본 규칙이라 **선택 편향이 없습니다.**
- 곡선의 **최고점**은 여러 기준 중 최고를 고른 값이라 그대로 주장하면 안 됩니다. 임계값을
  고정하려면 바깥 fold에서 골라야 합니다.
- 곡선이 **완만한 고원**이면 우연히 걸린 값일 가능성이 낮고, 한 점만 튀면 의심해야 합니다.

### 운영점 고르기

다수결에서 기준을 한 칸씩 낮출 때 **몇 건을 더 읽고 몇 건을 더 건지는지** 봅니다.


In [ ]:
base = sweep_df.iloc[majority - 1]
print(f'기준선: {base["기준"]} (다수결) — 표시 {base["표시"]:.0f}건, 놓침 {base["놓침"]:.0f}건')
print()
print(f'{"기준":<10}{"추가 표시":>10}{"덜 놓침":>10}{"1건당 추가 열람":>18}')
for i in range(majority - 2, -1, -1):
    row = sweep_df.iloc[i]
    more_read = row['표시'] - base['표시']
    fewer_missed = base['놓침'] - row['놓침']
    cost = more_read / fewer_missed if fewer_missed else float('nan')
    print(f'{row["기준"]:<10}{more_read:>10.0f}{fewer_missed:>10.0f}{cost:>18.1f}')


`1건당 추가 열람`이 이 선택의 환율입니다. 요구사항 하나 검토에 3분이 든다면 2.6건은 약 8분이고,
그 8분으로 계약 리스크 1건을 더 잡는 셈입니다. 그 교환이 남는지는 모델이 아니라 **업무가 답할**
**문제**입니다.

---

## 남은 조건

- 여기 값은 전부 **같은 924건**에서 나왔습니다. 앙상블 확정은 새 RFP에서 합니다.
- 임계값을 고정하려면 **바깥 fold에서 선택**해야 합니다 (§14 열린 질문).
- `finetune_runs.jsonl`에는 예측 라벨만 있고 **확률값이 없습니다.** soft voting과 확률 기반
  임계값 조정은 재실행이 필요합니다.
